In [ ]:
import numpy as np
import pandas as pd

from skillmodels.chs.filtered_states import get_filtered_states
from skillmodels.chs.maximization_inputs import get_maximization_inputs
from skillmodels.common.config import REGRESSION_VAULT, TEST_DATA_DIR
from skillmodels.common.simulate_data import simulate_dataset
from skillmodels.common.visualize_factor_distributions import (
    bivariate_density_contours,
    bivariate_density_surfaces,
    combine_distribution_plots,
    univariate_densities,
)
from skillmodels.test_data.model2 import MODEL2, MODEL2_CHS_OPTIONS

# How to visualize the distribution of latent factors

We show how to create kernel density plots for pairs of latent factors in two or three dimensions. As illustration we use the same example as in the [introductory tutorial](../getting_started/tutorial.ipynb). For more details on how to obtain filtered states, see that tutorial.

There are two kinds of data that the plotting functions consume:

1. Filtered states — point estimates of the latent factors for an empirical dataset.
2. Simulated states — a synthetic latent-factor panel from a parameterised model.

Below, we show how to produce both, how to visualise the distribution of latent factors given one dataset, and how to overlay two datasets (e.g. baseline vs. policy).

## Getting filtered states

In [ ]:
model = MODEL2
params = pd.read_csv(REGRESSION_VAULT / "one_stage_anchoring.csv")
params = params.set_index(["category", "period", "name1", "name2"])

data = pd.read_stata(TEST_DATA_DIR / "model2_simulated_data.dta")
data = data.set_index(["caseid", "period"])

filtered = get_filtered_states(model_spec=model, data=data, params=params)
states = filtered["anchored_states"]["states"]

## Plotting one dataset of states

In [ ]:
kde_plots = univariate_densities(
    model_spec=model,
    data=data,
    period=1,
    filtered_states=states,
)
contour_plots = bivariate_density_contours(
    model_spec=model,
    data=data,
    period=1,
    filtered_states=states,
)

In [ ]:
surface_plots = bivariate_density_surfaces(
    model_spec=model,
    data=data,
    period=1,
    filtered_states=states,
)

In [ ]:
fig = combine_distribution_plots(
    kde_plots=kde_plots,
    contour_plots=contour_plots,
    surface_plots=surface_plots,
)

In [ ]:
fig.show()

## Optional arguments of the plotting function

- You can omit the 3D plots by passing `surface_plots=None` to `combine_distribution_plots`.
- `n_points` controls the runtime/quality trade-off (grid points per dimension; default 50).
- `state_ranges` (dict keyed by factor name, values are DataFrames with columns `period`, `minimum`, `maximum`) lets you set axis limits manually.
- `layout_kwargs` is forwarded to Plotly `update_layout` for every subplot; `distplot_kwargs` / `contour_kwargs` go to the underlying traces.

## Simulated states with and without policy

A common application of skill-formation models is to simulate the effect of counterfactual policies. To visualise the effect of a policy on factor distributions, simulate one dataset with the policy and one without, then overlay them.

In [ ]:
sim_states = simulate_dataset(model_spec=model, params=params, data=data)[
    "anchored_states"
]["states"]

In [ ]:
policies = [
    {"period": 1, "factor": "fac1", "effect_size": 3.5, "standard_deviation": 0.0},
    {"period": 1, "factor": "fac2", "effect_size": 3.5, "standard_deviation": 0.0},
]

In [ ]:
sim_states_policy = simulate_dataset(
    model_spec=model,
    params=params,
    data=data,
    policies=policies,
)["anchored_states"]["states"]

## Plotting differences in distributions

Pass the simulated states as a dict (or list) of DataFrames; the plot helpers overlay one trace per scenario. 3D surface plots require a single DataFrame and don't support multi-scenario overlays.

In [ ]:
kde_plots = univariate_densities(
    model_spec=model,
    data=data,
    period=1,
    filtered_states={"baseline": sim_states, "subsidy": sim_states_policy},
)
contour_plots = bivariate_density_contours(
    model_spec=model,
    data=data,
    period=1,
    filtered_states={"baseline": sim_states, "subsidy": sim_states_policy},
)

In [ ]:
fig = combine_distribution_plots(kde_plots, contour_plots, None, showlegend=True)

In [ ]:
fig.show()

# Plotting with observed factors

Observed factors are columns in your dataset; pass `observed_factors=True` to include them. The filtered-states extraction is the same; the only change is which factor names you ask the plotter for.

In [ ]:
model = MODEL2.with_added_observed_factors("obs1")

In [ ]:
rng = np.random.default_rng(42)
data["obs1"] = rng.random(data.shape[0])

In [ ]:
params = get_maximization_inputs(
    model_spec=model, data=data, chs_options=MODEL2_CHS_OPTIONS
)["params_template"]
params["value"] = 0.1
filtered = get_filtered_states(model_spec=model, data=data, params=params)
states = filtered["anchored_states"]["states"]

In [ ]:
kde_plots = univariate_densities(
    model_spec=model,
    data=data,
    period=1,
    filtered_states=states,
    observed_factors=True,
)
contour_plots = bivariate_density_contours(
    model_spec=model,
    data=data,
    period=1,
    filtered_states=states,
    observed_factors=True,
)

In [ ]:
combine_distribution_plots(
    kde_plots=kde_plots,
    contour_plots=contour_plots,
    factor_order=["obs1", "fac1", "fac2"],
)